# Recommender System using Implicit ALS

## Objective

The goal of this project is to build a recommender system
based on implicit user–item interactions.

The model is trained using Alternating Least Squares (ALS)
and evaluated against a baseline popularity model.

# Setup & load data

In this section we import required libraries, set basic display options and random seed, load the Steam dataset CSV, and assign column names.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

SEED = 42
np.random.seed(SEED)

In [ ]:
DATA_PATH = "../data/steam-200k.csv"

df_raw = pd.read_csv(DATA_PATH, header=None)

df_raw.columns = ["user_id", "game", "behavior", "value", "unused"]

display(df_raw.head())
df_raw.info()

,user_id,game,behavior,value,unused
0,151603712,The Elder Scrolls V Skyrim,purchase,1.0,0
1,151603712,The Elder Scrolls V Skyrim,play,273.0,0
2,151603712,Fallout 4,purchase,1.0,0
3,151603712,Fallout 4,play,87.0,0
4,151603712,Spore,purchase,1.0,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 5 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   user_id   200000 non-null  int64  
 1   game      200000 non-null  object 
 2   behavior  200000 non-null  object 
 3   value     200000 non-null  float64
 4   unused    200000 non-null  int64  
dtypes: float64(1), int64(2), object(2)
memory usage: 7.6+ MB


The dataset contains 200,000 interaction records describing activity of Steam users. Each row corresponds to a single user–game event and includes:
- user_id
- game: game title
- behavior: interaction type (purchase or play)
- value: numeric value associated with the event


  for purchase: typically 1.0


  for play: playtime in hours
- unused: unused column that can be ignored

**This is an implicit feedback dataset because user preference is inferred from behavioral signals such as playtime and purchases, rather than explicit opinions or star ratings provided by users.**

# Sanity checks & quick EDA

Here we validate the dataset structure: number of rows, unique users/items, behavior types (purchase/play), missing values, and basic statistics for play hours.

In [ ]:
print("Rows:", len(df_raw))
print("Unique users:", df_raw["user_id"].nunique())
print("Unique games:", df_raw["game"].nunique())

print("\nBehavior value counts:")
print(df_raw["behavior"].value_counts(dropna=False))

print("\nMissing values per column:")
print(df_raw.isna().sum())

display(df_raw[df_raw["behavior"] == "purchase"].head())
display(df_raw[df_raw["behavior"] == "play"].head())

Rows: 200000
Unique users: 12393
Unique games: 5155

Behavior value counts:
behavior
purchase    129511
play         70489
Name: count, dtype: int64

Missing values per column:
user_id     0
game        0
behavior    0
value       0
unused      0
dtype: int64


,user_id,game,behavior,value,unused
0,151603712,The Elder Scrolls V Skyrim,purchase,1.0,0
2,151603712,Fallout 4,purchase,1.0,0
4,151603712,Spore,purchase,1.0,0
6,151603712,Fallout New Vegas,purchase,1.0,0
8,151603712,Left 4 Dead 2,purchase,1.0,0


,user_id,game,behavior,value,unused
1,151603712,The Elder Scrolls V Skyrim,play,273.0,0
3,151603712,Fallout 4,play,87.0,0
5,151603712,Spore,play,14.9,0
7,151603712,Fallout New Vegas,play,12.1,0
9,151603712,Left 4 Dead 2,play,8.9,0


The dataset contains 12,393 users and 5,155 unique games.

We observe two interaction types: purchase (129,511 rows) and play (70,489 rows). The fact that purchase events significantly outnumber play events suggests that not every purchased game is actually played.

In [ ]:
play_df = df_raw[df_raw["behavior"] == "play"].copy()

play_df["hours"] = pd.to_numeric(play_df["value"], errors="coerce")

display(play_df["hours"].describe(percentiles=[.5, .9, .95, .99]))

print("Hours == 0:", (play_df["hours"] == 0).sum())
print("Hours <= 1:", (play_df["hours"] <= 1).sum())
print("Hours >= 100:", (play_df["hours"] >= 100).sum())

,hours
count,70489.000000
mean,48.878063
std,229.335236
min,0.100000
50%,4.500000
90%,76.000000
95%,189.000000
99%,908.000000
max,11754.000000


Hours == 0: 0
Hours <= 1: 17850
Hours >= 100: 5781


There are no zero-hour play events (Hours == 0: 0), so the dataset records playtime only when it is positive.

Playtime is highly right-skewed: the median is 4.5 hours, while the mean is 48.9 hours, indicating that a small fraction of users accumulate extremely large playtimes.

The long tail is substantial: 99% of play events are below 908 hours, yet the maximum reaches 11,754 hours, which strongly suggests the presence of heavy outliers.

# Build an implicit preference signal (purchase + playtime)

This section converts raw events into a single implicit preference score per (user, game). We treat purchase as a weak positive signal and playtime as a stronger signal (log-scaled). Then we aggregate to one row per user-game pair.

In [ ]:
W_PURCHASE = 0.4
W_PLAY = 3.0

df = df_raw.copy()

df["hours"] = np.where(
    df["behavior"] == "play",
    pd.to_numeric(df["value"], errors="coerce"),
    0.0
)

df["purchase_flag"] = (df["behavior"] == "purchase").astype(int)

df["play_signal"] = np.log1p(df["hours"])

df["purchase_signal"] = df["purchase_flag"]

df["pref_row"] = W_PLAY * df["play_signal"] + W_PURCHASE * df["purchase_signal"]

display(df.head())

,user_id,game,behavior,value,unused,hours,purchase_flag,play_signal,purchase_signal,pref_row
0,151603712,The Elder Scrolls V Skyrim,purchase,1.0,0,0.0,1,0.000000,1,0.400000
1,151603712,The Elder Scrolls V Skyrim,play,273.0,0,273.0,0,5.613128,0,16.839384
2,151603712,Fallout 4,purchase,1.0,0,0.0,1,0.000000,1,0.400000
3,151603712,Fallout 4,play,87.0,0,87.0,0,4.477337,0,13.432010
4,151603712,Spore,purchase,1.0,0,0.0,1,0.000000,1,0.400000


In [ ]:
interactions = (
    df.groupby(["user_id", "game"], as_index=False)
      .agg(
          pref=("pref_row", "sum"),
          hours=("hours", "sum"),
          purchased=("purchase_flag", "max")
      )
)

print("Interactions rows:", len(interactions))
display(interactions.head())

display(interactions["pref"].describe(percentiles=[.5, .9, .95, .99]))

Interactions rows: 128804


,user_id,game,pref,hours,purchased
0,5250,Alien Swarm,5.724857,4.9,1
1,5250,Cities Skylines,15.330201,144.0,1
2,5250,Counter-Strike,0.400000,0.0,1
3,5250,Counter-Strike Source,0.400000,0.0,1
4,5250,Day of Defeat,0.400000,0.0,1


,pref
count,128804.000000
mean,3.772660
std,4.717334
min,0.400000
50%,1.187093
90%,10.889523
95%,13.797724
99%,19.313346
max,28.516102


We compress multiple raw events into a single user–game interaction with a unified implicit preference score pref. This is necessary to build a standard user–item matrix for recommendation models.

The minimum pref value equals 0.4, which corresponds to purchase-only pairs (bought but not played).

# Data filtering

To make models more stable, we remove extremely sparse users/items.

In [ ]:
MIN_USERS_PER_GAME = 5
MIN_GAMES_PER_USER = 3

game_counts = interactions.groupby("game")["user_id"].nunique()
user_counts = interactions.groupby("user_id")["game"].nunique()

keep_games = set(game_counts[game_counts >= MIN_USERS_PER_GAME].index)
keep_users = set(user_counts[user_counts >= MIN_GAMES_PER_USER].index)

inter_filt = interactions[
    interactions["game"].isin(keep_games) & interactions["user_id"].isin(keep_users)
].copy()

print("Before:", interactions.shape, "After:", inter_filt.shape)
print("Users:", inter_filt["user_id"].nunique(), "Games:", inter_filt["game"].nunique())

Before: (128804, 5) After: (115408, 5)
Users: 5216 Games: 2669


We removed the sparsest parts of the dataset by keeping only games with at least 5 users and users with at least 3 games.

# Create user/item index mappings

Here we map user IDs and game titles to contiguous integer indices and store both forward and reverse mappings.

In [ ]:
data = inter_filt.copy()

user_ids = data["user_id"].unique()
game_ids = data["game"].unique()

user_to_idx = {u: i for i, u in enumerate(user_ids)}
idx_to_user = {i: u for u, i in user_to_idx.items()}

game_to_idx = {g: i for i, g in enumerate(game_ids)}
idx_to_game = {i: g for g, i in game_to_idx.items()}

data["user_idx"] = data["user_id"].map(user_to_idx)
data["game_idx"] = data["game"].map(game_to_idx)

print("n_users:", len(user_ids), "n_games:", len(game_ids))
display(data.head())

n_users: 5216 n_games: 2669


,user_id,game,pref,hours,purchased,user_idx,game_idx
0,5250,Alien Swarm,5.724857,4.9,1,0,0
1,5250,Cities Skylines,15.330201,144.0,1,0,1
2,5250,Counter-Strike,0.400000,0.0,1,0,2
3,5250,Counter-Strike Source,0.400000,0.0,1,0,3
4,5250,Day of Defeat,0.400000,0.0,1,0,4


# Train/test split

In [ ]:
TEST_RATIO = 0.2
MIN_TEST_ITEMS_PER_USER = 1

rng = np.random.default_rng(SEED)

def split_user_interactions(df_ui, test_ratio=0.2, min_test=1):
    train_parts = []
    test_parts = []

    for user_idx, grp in df_ui.groupby("user_idx"):
        n = len(grp)
        if n < 2:
            train_parts.append(grp)
            continue

        test_size = max(min_test, int(np.floor(n * test_ratio)))
        test_size = min(test_size, n - 1)

        test_idx = rng.choice(grp.index.values, size=test_size, replace=False)
        test_parts.append(grp.loc[test_idx])
        train_parts.append(grp.drop(index=test_idx))

    train_df = pd.concat(train_parts).reset_index(drop=True)
    test_df = pd.concat(test_parts).reset_index(drop=True) if len(test_parts) else pd.DataFrame(columns=df_ui.columns)
    return train_df, test_df

train_df, test_df = split_user_interactions(data, test_ratio=TEST_RATIO, min_test=MIN_TEST_ITEMS_PER_USER)

print("Train interactions:", train_df.shape)
print("Test interactions:", test_df.shape)

overlap = pd.merge(
    train_df[["user_idx","game_idx"]],
    test_df[["user_idx","game_idx"]],
    on=["user_idx","game_idx"],
    how="inner"
)
print("Train/Test overlap pairs:", len(overlap))

Train interactions: (93154, 7)
Test interactions: (22254, 7)
Train/Test overlap pairs: 0


We split the data into train and test in a user-based way. For every user we randomly put about 20% of their games into the test set and keep the rest in the train set. We also make sure each user has at least one game in train, so the model has some history to learn from.
At the end we check that there are no (user, game) pairs that appear in both train and test (overlap = 0), so there is no data leakage.

In [ ]:
from collections import defaultdict

def get_user_test_items(test_df):
    """Create dict: user_idx -> set of game_idx from the test set."""
    user_test = defaultdict(set)
    for u, i in zip(test_df["user_idx"], test_df["game_idx"]):
        user_test[int(u)].add(int(i))
    return user_test

user_test_items = get_user_test_items(test_df)

print("Users with at least 1 test item:", len(user_test_items))
print("Total test interactions:", len(test_df))

Users with at least 1 test item: 5212
Total test interactions: 22254


^^^^^^^

In this step we prepare the test data for future evaluation. Our evaluation metrics (Precision@K and Recall@K) need, for each user, the list of games that were hidden in the test set.

# Baseline: global popularity

We build a popularity-based recommender using the training set only. This serves as a baseline and as the fallback strategy for true cold-start users with no history.

In [ ]:
popularity = (
    train_df.groupby("game_idx", as_index=False)
            .agg(pop_score=("pref", "sum"),
                 user_count=("user_idx", "nunique"))
            .sort_values(["pop_score", "user_count"], ascending=False)
)

TOP_POP_GAMES = popularity["game_idx"].tolist()

def recommend_popular(k=10):
    return TOP_POP_GAMES[:k]

top10 = recommend_popular(10)
print("Top-10 popular game titles:")
for i, gid in enumerate(top10, 1):
    print(i, idx_to_game[gid])

Top-10 popular game titles:
1 Dota 2
2 Counter-Strike Global Offensive
3 Team Fortress 2
4 The Elder Scrolls V Skyrim
5 Counter-Strike Source
6 Garry's Mod
7 Left 4 Dead 2
8 Counter-Strike
9 Sid Meier's Civilization V
10 Unturned


This baseline recommends the same Top-k games to everyone, based on the training data only. It is useful as:
- a simple reference point to compare with the ALS model, and
- ma fallback for true cold-start users (new users with no history).

In our output we see well-known, widely played games (e.g., Dota 2, CS:GO, Team Fortress 2), which is expected for a popularity-based method.

# Build sparse matrices

We convert interactions into a sparse user-item matrix.

In [ ]:
from scipy.sparse import coo_matrix, csr_matrix

n_users = len(user_to_idx)
n_items = len(game_to_idx)

def build_user_item_matrix(df_ui, n_users, n_items):
    rows = df_ui["user_idx"].to_numpy()
    cols = df_ui["game_idx"].to_numpy()
    vals = df_ui["pref"].to_numpy().astype(np.float32)

    mat = coo_matrix((vals, (rows, cols)), shape=(n_users, n_items)).tocsr()
    return mat

train_mat = build_user_item_matrix(train_df, n_users, n_items)
test_mat = build_user_item_matrix(test_df, n_users, n_items)

train_mat.shape, test_mat.shape

((5216, 2669), (5216, 2669))

Most users played/bought only a few games, so the matrix is mostly empty. To save memory and make training faster, we store it as a sparse matrix (CSR).

# Model: Implicit ALS

We train an Implicit Alternating Least Squares model on the implicit feedback matrix.

In [ ]:
from implicit.als import AlternatingLeastSquares

ALPHA = 20.0

train_conf_ui = train_mat.copy().tocsr().astype(np.float32)
train_conf_ui.data = 1.0 + ALPHA * train_conf_ui.data

ALS_FACTORS = 64
ALS_REG = 0.05
ALS_ITERS = 20

als_model = AlternatingLeastSquares(
    factors=ALS_FACTORS,
    regularization=ALS_REG,
    iterations=ALS_ITERS,
    random_state=SEED,
    use_gpu=False
)

als_model.fit(train_conf_ui)

print("ALS trained.")
print("user_factors:", als_model.user_factors.shape)
print("item_factors:", als_model.item_factors.shape)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 4.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

ALS trained.
user_factors: (5216, 64)
item_factors: (2669, 64)


We trained an ALS model on implicit feedback. We changed the train matrix into a confidence matrix using alpha. The model learned vectors for users and games.
Now we can recommend new games for a user based on these vectors.

# Warm-Start recommendations

We generate personalized Top-N recommendations for users with history using the trained ALS model, while filtering out items already seen in training.

In [ ]:
def recommend_als_for_user(user_idx, k=10):
    """Return a list of (game_idx, score) recommendations for a known user_idx using ALS."""
    user_items_row = train_conf_ui[user_idx]

    recs = als_model.recommend(
        userid=user_idx,
        user_items=user_items_row,
        N=k,
        filter_already_liked_items=True
    )
    if isinstance(recs, tuple) and len(recs) == 2:
        item_ids, scores = recs
        pairs = list(zip(item_ids.tolist(), scores.tolist()))
    else:
        pairs = [(int(i), float(s)) for i, s in recs]

    pairs = [(i, s) for i, s in pairs if 0 <= i < n_items]
    return pairs[:k]

In [ ]:
def show_user_profile_and_recs(user_idx, k=10, top_n_played=10):
    """Print top played games (by hours) and ALS recommendations for one user."""
    user_id = idx_to_user[user_idx]
    user_rows = train_df[train_df["user_idx"] == user_idx].copy()

    print("=" * 80)
    print("USER PROFILE (train history)")
    print(f"user_idx: {user_idx} | original user_id: {user_id} | train interactions: {len(user_rows)}")
    print("=" * 80)

    top_by_hours = user_rows.sort_values("hours", ascending=False).head(top_n_played)

    print("\nTop-10 most played games (train)  game | hours | pref")
    for rank, r in enumerate(top_by_hours.itertuples(index=False), 1):
        print(f"{rank:>2}. {r.game} | hours: {float(r.hours):>7.1f} | pref: {float(r.pref):>6.2f}")

    recs = recommend_als_for_user(user_idx, k=k)
    print("\nTop recommendations from ALS  game | score")
    for rank, (gid, score) in enumerate(recs, 1):
        print(f"{rank:>2}. {idx_to_game[gid]} | score: {float(score):.4f}")

    print()


N_USERS_TO_SHOW = 4

sample_users = (
    train_df["user_idx"]
    .drop_duplicates()
    .sample(N_USERS_TO_SHOW)
    .tolist()
)

for u in sample_users:
    show_user_profile_and_recs(int(u), k=10, top_n_played=10)

USER PROFILE (train history)
user_idx: 1897 | original user_id: 89867101 | train interactions: 3

Top-10 most played games (train)  game | hours | pref
 1. Counter-Strike Source | hours:   632.0 | pref:  19.75
 2. Day of Defeat Source | hours:     0.0 | pref:   0.40
 3. Half-Life 2 Deathmatch | hours:     0.0 | pref:   0.40

Top recommendations from ALS  game | score
 1. Half-Life 2 Lost Coast | score: 0.9002
 2. Half-Life 2 | score: 0.6313
 3. Half-Life 2 Episode One | score: 0.4150
 4. Half-Life Deathmatch Source | score: 0.3611
 5. Half-Life 2 Episode Two | score: 0.3456
 6. Half-Life Source | score: 0.2983
 7. Left 4 Dead | score: 0.2264
 8. Counter-Strike Condition Zero | score: 0.2190
 9. Portal | score: 0.2081
10. Counter-Strike Condition Zero Deleted Scenes | score: 0.2025

USER PROFILE (train history)
user_idx: 877 | original user_id: 40652120 | train interactions: 5

Top-10 most played games (train)  game | hours | pref
 1. Counter-Strike Source | hours:    51.0 | pref:  12.2

# Cold-Start (zero history) recommendation function

We implement a unified recommend() function. If the user is unknown (cold-start, no history), we return the popularity-based fallback recommendations.

In [ ]:
def recommend_cold_start(k=10):
    """Cold-start recommendations (zero history) using global popularity."""
    top_game_ids = recommend_popular(k)
    return [idx_to_game[gid] for gid in top_game_ids]

def simulate_cold_start_for_existing_user(existing_user_id, k=10):
    """
    Cold-start simulation
    We take a real user_id but we pretend the user is new. So we return popularity-based recommendations
    """
    print("=" * 70)
    print("COLD-START SIMULATION")
    print(f"Existing user_id treated as new: {existing_user_id}")
    print("=" * 70)

    titles = recommend_cold_start(k=k)

    df_out = pd.DataFrame({
        "rank": range(1, len(titles) + 1),
        "recommended_game": titles
    })

    display(df_out)

example_user_id = int(train_df["user_id"].sample(1, random_state=SEED).iloc[0])
simulate_cold_start_for_existing_user(example_user_id, k=10)

COLD-START SIMULATION
Existing user_id treated as new: 22301321


,rank,recommended_game
0,1,Dota 2
1,2,Counter-Strike Global Offensive
2,3,Team Fortress 2
3,4,The Elder Scrolls V Skyrim
4,5,Counter-Strike Source
5,6,Garry's Mod
6,7,Left 4 Dead 2
7,8,Counter-Strike
8,9,Sid Meier's Civilization V
9,10,Unturned


In this cold-start simulation we treat an existing user as a completely new user with zero history, so the system cannot personalize recommendations. As expected, the output is the same global popularity list as our baseline.

# Evaluation

In [ ]:
def precision_recall_at_k(model, train_conf_ui, user_test_items, k=10, max_users=2000):
    users = [
        u for u in user_test_items.keys()
        if len(user_test_items[u]) > 0 and 0 <= u < n_users
    ]

    rng_local = np.random.default_rng(SEED)
    if len(users) > max_users:
        users = rng_local.choice(users, size=max_users, replace=False).tolist()

    precisions, recalls = [], []

    for u in users:
        test_items = user_test_items[u]

        user_items_row = train_conf_ui[u]

        recs = model.recommend(
            userid=u,
            user_items=user_items_row,
            N=k,
            filter_already_liked_items=True
        )

        if isinstance(recs, tuple) and len(recs) == 2:
            rec_items = [int(i) for i in recs[0]]
        else:
            rec_items = [int(i) for i, _ in recs]

        rec_items = [i for i in rec_items if 0 <= i < n_items]

        hits = len(set(rec_items) & set(test_items))

        precisions.append(hits / k)

        recalls.append(hits / len(test_items))

    return float(np.mean(precisions)), float(np.mean(recalls)), len(users)


In [ ]:
K = 10
p_at_k, r_at_k, n_eval_users = precision_recall_at_k(
    als_model, train_conf_ui, user_test_items, k=K, max_users=2000
)

print("Evaluation results")
print("Evaluated users:", n_eval_users)
print(f"Avg hits in Top-{K}:", round(p_at_k * K, 2))
print(f"Precision@{K}:", round(p_at_k, 4))
print(f"Recall@{K}:", round(r_at_k, 4))

Evaluation results
Evaluated users: 2000
Avg hits in Top-10: 0.9
Precision@10: 0.0897
Recall@10: 0.4269


Precision is not very high because the catalog is large and Top-10 is short, so most recommended games will not match the hidden test set
Recall is much higher, which means the model is able to recover a good part of the hidden games in the test set
Overall, the results suggest that the ALS model learns user preferences from playtime and can recommend relevant games better than random guessing.

--------------------------------------------------------------------------------

In [ ]:
def one_user_case_study(k=10, max_show_test=10, random_state=44):
    """
    Pick one user who has test items and show
    hidden test games and Top-K ALS recommendations with HIT markers
    """
    users_with_test = [u for u in user_test_items.keys() if len(user_test_items[u]) > 0]
    user_idx = int(np.random.default_rng(random_state).choice(users_with_test))

    test_items = user_test_items[user_idx]

    recs = recommend_als_for_user(user_idx, k=k)
    rec_items = [gid for gid, _ in recs]

    hits = set(rec_items) & set(test_items)

    print("=" * 80)
    print("ONE USER CASE STUDY")
    print(f"user_idx: {user_idx} | original user_id: {idx_to_user[user_idx]}")
    print(f"test items: {len(test_items)} | hits@{k}: {len(hits)}")
    print("=" * 80)

    print("\nHidden test games (sample)")
    for gid in list(test_items)[:max_show_test]:
        print(" -", idx_to_game[gid])

    print(f"\nTop-{k} ALS recommendations (HIT means the game is in the hidden test set)")
    for rank, (gid, score) in enumerate(recs, 1):
        mark = "HIT" if gid in test_items else ""
        print(f"{rank:>2}. {idx_to_game[gid]} | score: {float(score):.4f} {mark}")

one_user_case_study(k=10)

ONE USER CASE STUDY
user_idx: 3476 | original user_id: 175263669
test items: 6 | hits@10: 2

Hidden test games (sample)
 - Fallen Earth
 - Dota 2
 - PlanetSide 2
 - Warframe
 - Starbound
 - Unturned

Top-10 ALS recommendations (HIT means the game is in the hidden test set)
 1. Dead Island Epidemic | score: 1.0905 
 2. Unturned | score: 0.9152 HIT
 3. Survarium | score: 0.9117 
 4. Trove | score: 0.8796 
 5. Starbound | score: 0.8213 HIT
 6. HAWKEN | score: 0.7336 
 7. Toribash | score: 0.7304 
 8. APB Reloaded | score: 0.7042 
 9. Infestation Survivor Stories | score: 0.6947 
10. Only If | score: 0.6864 


In this step we show a simple case study for one user to make the evaluation easier to understand. We randomly pick a user who has at least one hidden interaction in the test set.
Then we print two lists:
- hidden test games: games that were removed from this user's history and kept only for evaluation
- top-K ALS recommendations: games recommended by the ALS model

We also mark recommendations as HIT if the recommended game appears in the hidden test set.

This helps us visually check if the model is able to recover games the user actually played but were hidden during training.

--------------------------------------------------------------------------------

In this section we compare our ALS recommender with a simple baseline based on global popularity.

In [ ]:
def precision_recall_popularity_at_k(user_test_items, k=10, max_users=2000):
    """
    Evaluate the popularity baseline with Precision@K and Recall@K
    Popularity baseline recommends the same global Top-K games to every user
    """
    users = [
        u for u in user_test_items.keys()
        if len(user_test_items[u]) > 0 and 0 <= u < n_users
    ]

    rng_local = np.random.default_rng(SEED)
    if len(users) > max_users:
        users = rng_local.choice(users, size=max_users, replace=False).tolist()

    pop_items = recommend_popular(k)

    precisions, recalls = [], []
    for u in users:
        test_items = user_test_items[u]

        hits = len(set(pop_items) & set(test_items))
        precisions.append(hits / k)
        recalls.append(hits / len(test_items))

    return float(np.mean(precisions)), float(np.mean(recalls)), len(users)


K = 10

p_als, r_als, n_als = precision_recall_at_k(
    als_model, train_conf_ui, user_test_items, k=K, max_users=2000
)

p_pop, r_pop, n_pop = precision_recall_popularity_at_k(
    user_test_items, k=K, max_users=2000
)

compare_df = pd.DataFrame([
    {
        "model": "ALS",
        "K": K,
        "Precision@K": round(p_als, 4),
        "Recall@K": round(r_als, 4),
        "Avg hits in Top-K": round(p_als * K, 2),
        "Evaluated users": n_als
    },
    {
        "model": "Popularity baseline (cold-start)",
        "K": K,
        "Precision@K": round(p_pop, 4),
        "Recall@K": round(r_pop, 4),
        "Avg hits in Top-K": round(p_pop * K, 2),
        "Evaluated users": n_pop
    }
])

display(compare_df)

,model,K,Precision@K,Recall@K,Avg hits in Top-K,Evaluated users
0,ALS,10,0.0897,0.4269,0.90,2000
1,Popularity baseline (cold-start),10,0.0377,0.1704,0.38,2000


The ALS model significantly outperforms the popularity baseline
across both Precision@10 and Recall@10.

- Precision@10 improved from 0.0377 to 0.0897.
- Recall@10 improved from 0.1704 to 0.4269.
- The average number of hits in Top-10 recommendations more than doubled.

This indicates that collaborative filtering captures personalized user preferences
more effectively than global popularity-based recommendations.

However, the popularity model remains useful as a fallback strategy
for true cold-start users with no interaction history.